# Complete Energy-Profile Comparison — LaTeX tables & boxplot

Same distance-to-real feature scorecard as `results_energy_profile_compare`, but
adds:
1. the **aggregated** scorecard as LaTeX,
2. one scorecard **per process** (separated) as LaTeX, and
3. a **boxplot across all processes** (per method), styled like the sMAE boxplot
   in `results_latex_table`.

Methods: Alpha, Combined-best, Budget (process types, best duration approach per
process), plus Schedule-direct and Profile-generator.

**Metric — paired relative error, not a Wasserstein distance.** For each
`(process, case, sensor)` unit and each per-curve feature `f`:

    err = |f(predicted case) - f(real case)| / mean|f(real)|

i.e. every simulated case is compared against *its own* real counterpart, and the
error is normalised by the typical real magnitude of that sensor so it is
comparable across sensors. Tables report the **median over units**;
**lower = closer to real**. (An earlier version compared the *distribution* of
each feature over cases via a Wasserstein distance — that was unpaired and
weighted by sensor count, and was replaced when the aggregation unit became
`(process, case, sensor)`.)

All columns are paired relative errors of a per-curve shape scalar
(Total, Peak, …).

In [10]:
# ── Config ────────────────────────────────────────────────────────────────────
import os, glob, warnings
import numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import wasserstein_distance
from IPython.display import display, Markdown
import matplotlib.pyplot as plt, matplotlib.colors as mcolors
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')

EXPERIMENT     = 1000

# ══ METHODS — turn any row of every table/plot on or off here ═══════════════
# Order here is the order they appear in every table and boxplot.
METHODS = {
    'Baseline':          True,   # per-SENSOR median curve (naive curve generator)
    'Alpha':             True,   # alpha-miner net        + CURVE_APPROACH
    'Combined-best':     True,   # best discovered net    + CURVE_APPROACH
    'Budget':            True,   # best net + duration budgeting + CURVE_APPROACH
    'Schedule-direct':   True,   # real schedule, curves taken directly
    'Profile-generator': True,   # stochastic profile generator
}
# ═══════════════════════════════════════════════════════════════════════════

# ══ FEATURES — turn any column of every table/plot on or off here ══════════
# Order here is the column order in every table.
FEATURE_TOGGLE = {
    'total':     True,    # sum of the curve's samples
    'auc':       False,   # area under the curve, trapezoidal integral of v dt
    'peak':      True,    # max draw
    'mean':      True,
    'median':    False,
    'std':       True,
    'time':      False,   # paired per-case W1 of the value-weighted time distribution
                      # (WHEN the energy is drawn, as a fraction of the case's own
                      # real duration) -- the timing axis, orthogonal to 'w1_load'
    'w1_load':   False,   # paired per-case W1 between the real and the predicted LOAD-LEVEL
                      # distribution (the load-duration curve), expressed as a fraction of
                      # the sensor's typical real load. Sum/Max/Mean/Std are four single
                      # moments of that distribution; this is the whole distribution
    'ac1':       False,   # lag-1 autocorrelation (smoothness)
    'roughness': True,    # mean |dv| (jaggedness) -- texture check: a method can hit
                      # the right Std by adding noise; only this separates that from
                      # reproducing the real dynamics
    'zero_frac': False,   # share of samples at the idle floor (duty cycle)
}
# ═══════════════════════════════════════════════════════════════════════════

SHOW_OVERALL     = True   # add an 'Overall' column = average across the metric columns
SORT_BY_OVERALL  = True   # order the rows best-to-worst by it

CURVE_APPROACH = 'ml_step_dtw_smooth'
# Baseline row: same simulated process as BASELINE_PROCESS_TYPE, but the naive
# per-sensor median curve instead of CURVE_APPROACH. Holding the process model
# fixed means the Baseline vs. that row difference is attributable to the curve
# generator alone. 'baseline' is stored unsuffixed by the pipeline.
BASELINE_CURVE_APPROACH = 'baseline'
BASELINE_PROCESS_TYPE   = 'Budget'

DURATION_MODE  = 'best_by_mae'    # per-process best time prediction, chosen on SELECTION_SPLIT
                                  # (or fix it: 'baseline' / 'ml_global' / 'ml_local')
DURATION_SELECT_METRIC = 'duration_metrics_activity_duration_mae'

# Derived views of METHODS (kept so the rest of the notebook reads unchanged)
PROCESS_TYPES = {k: METHODS.get(k, False) for k in ('Alpha', 'Combined-best', 'Budget')}
EXTRA_METHODS = {k: METHODS.get(k, False) for k in ('Schedule-direct', 'Profile-generator')}
SHOW_BASELINE = METHODS.get('Baseline', False)
# Model selection ALWAYS happens on TRAIN, independently of SPLIT (which only
# controls what the tables report). Selecting on the split being reported would
# let a model be chosen for fitting the evaluation data.
SELECTION_SPLIT = 'train'

SELECTION_METRIC_BASES = [
    'conformance_metrics_fitness',
    'conformance_metrics_precision',
    'conformance_metrics_generalization',
    'conformance_metrics_simplicity',
]
# The four selection metrics are QUALITY scores (higher = better), so the best
# model is the argmax of their average -- not the argmin used when the criterion
# was expressed as errors.
SELECTION_HIGHER_IS_BETTER = True
COMPOSITE_INCLUDE_TIME = True   # fold W1(time) into the composite boxplot score
SAVE_LATEX = True

results_root = Path('..') / 'results'
runs = sorted([d for d in results_root.iterdir()
               if d.is_dir() and d.name.startswith(f'experiment_{EXPERIMENT}_')])
assert runs, f'No runs for experiment {EXPERIMENT}'
RUN = runs[-1]
print('Using run:', RUN.name)
SCHEDULE_SERIES = {'schedule': 'Schedule-direct', 'stochastic': 'Profile-generator'}
def _suffix_for(approach):
    # The pipeline writes the 'baseline' approach without a suffix.
    return '' if approach in ('baseline', '', None) else f'_{approach}'
_curve_suffix   = _suffix_for(CURVE_APPROACH)
_baseline_suffix = _suffix_for(BASELINE_CURVE_APPROACH)

Using run: experiment_1000_20260729_201444


In [11]:
# ── Per-curve features ───────────────────────────────────────────────────────
def curve_features(v, t=None):
    """Per-curve scalars. `t` is the curve's t_minutes axis.

    Two size measures are emitted; FEATURE_TOGGLE picks which is reported.
      'total' -- plain sum of the samples (the reported default).
      'auc'   -- trapezoidal integral of v dt, the true area in kW*min.
    They differ because sum(v) implicitly treats every sample as one minute
    wide and assumes all series share a grid. Measured median dt is 1.000 min
    for `real`, ~1.04 for the petri-net predictions and ~1.21 for the schedule
    / stochastic series, so 'total' carries a ~4% / ~21% grid-dependent offset
    on top of the prediction error, while 'auc' does not. 'auc' falls back to
    the sum when no time axis is available.
    """
    v = np.asarray(v, float)
    if t is not None:
        t = np.asarray(t, float)
        m = np.isfinite(v) & np.isfinite(t)
        v, t = v[m], t[m]
    else:
        v = v[np.isfinite(v)]
    if v.size < 4: return None
    auc = float(np.trapz(v, t)) if (t is not None and t.size == v.size) else float(np.nansum(v))
    rng = np.nanmax(v) - np.nanmin(v)
    zero = np.mean((v - np.nanmin(v)) <= (0.02 * rng if rng > 0 else 1e-9))
    ac1 = pd.Series(v).autocorr(lag=1)
    return {'total': float(np.nansum(v)), 'auc': auc, 'peak': float(np.nanmax(v)),
            'mean': float(np.nanmean(v)), 'median': float(np.nanmedian(v)),
            'std': float(np.nanstd(v)), 'ac1': float(ac1) if np.isfinite(ac1) else np.nan,
            'roughness': float(np.nanmean(np.abs(np.diff(v)))), 'zero_frac': float(zero)}

# 'time' and 'w1_load' are NOT per-curve shape features but PAIRED per-case
# metrics (see w1_time_case / w1_load_case below). They are carried in the same
# grid so each shows up as one more column in every table / boxplot;
# VALUE_FEATURES is what curve_features emits.
_ALL_VALUE_FEATURES  = ['total', 'auc', 'peak', 'mean', 'median', 'std', 'ac1', 'roughness', 'zero_frac']
_ALL_PAIRED_FEATURES = ['time', 'w1_load']
# curve_features() keeps computing all of them (cheap, and keeps the toggle
# reversible); FEATURE_TOGGLE decides what actually reaches the tables.
VALUE_FEATURES  = [f for f in _ALL_VALUE_FEATURES if FEATURE_TOGGLE.get(f)]
PAIRED_FEATURES = [f for f in _ALL_PAIRED_FEATURES if FEATURE_TOGGLE.get(f)]
FEATURES        = PAIRED_FEATURES + VALUE_FEATURES
INCLUDE_TIME    = FEATURE_TOGGLE.get('time', False)
INCLUDE_W1_LOAD = FEATURE_TOGGLE.get('w1_load', False)
INCLUDE_PAIRED  = bool(PAIRED_FEATURES)
FEAT_LABEL = {'time':'W1 (time)','w1_load':'W1 (load)','total':'Sum','auc':'Area (AUC)','peak':'Max',
              'mean':'Mean','median':'Median',
              'std':'Std','ac1':'AC1','roughness':'Roughness','zero_frac':'Zero-frac'}

def features_long(df_curves, method_label):
    rows = []
    for (sen, cid), g in df_curves.groupby(['sensor', 'case_id']):
        gs = g.sort_values('t_minutes')
        f = curve_features(gs['value'].to_numpy(), gs['t_minutes'].to_numpy())
        if f:
            f.update(sensor=sen, case_id=cid, series=method_label); rows.append(f)
    return rows

# ── W1(time): the timing metric from `results_latex_table` ───────────────────
# For ONE case: compare *when* the energy is drawn -- the value-weighted
# distribution over time -- in real vs. predicted, with time normalized by that
# case's OWN real duration (so a case that simply runs longer isn't penalized
# for its scale, only for shifting its energy around inside the case).
# Degenerate cases are skipped (NaN) instead of exploding the ratio, mirroring
# the `_cv_ok` / degenerate-scale guards used for the value features.
def w1_time_case(t_real, v_real, t_pred, v_pred):
    w_real = np.clip(v_real, 0, None); w_pred = np.clip(v_pred, 0, None)
    if w_real.sum() <= 0 or w_pred.sum() <= 0: return np.nan     # all-zero curve
    dur = float(np.nanmax(t_real))
    if not np.isfinite(dur) or dur <= 1e-6: return np.nan        # zero-length case
    return float(wasserstein_distance(t_real / dur, t_pred / dur,
                                      u_weights=w_real, v_weights=w_pred))

# ── W1(load): the same comparison, on the AMPLITUDE axis ─────────────────────
# For ONE case: compare *how much of the case is spent at each load level* --
# the load-duration curve read as a distribution over kW -- in real vs.
# predicted. W1 is the average distance a unit of that distribution has to be
# moved along the kW axis to turn one into the other, so it is in kW and is
# divided by the sensor's typical real load in the units cell below.
#
# Why it earns a column next to Sum / Max / Mean / Std: those are four single
# moments of exactly this distribution, and a method can land all four while
# spending its time at the wrong load levels -- e.g. flipping between idle and
# full instead of running steadily at half load matches Mean and Sum, and can
# be made to match Std, but not the distribution. W1 is blind to ORDER, though:
# shuffling a curve in time leaves it unchanged, which is what W1(time), AC1
# and Roughness are there for.
#
# Each sample is weighted by the time it stands for (its dt) rather than
# counting 1, so a series on a coarser grid is not silently down-weighted --
# the same grid-dependence that 'auc' exists to dodge (real dt = 1.00 min,
# petri-net ~1.04, schedule / stochastic ~1.21).
def _dt_weights(t, v):
    """Minutes each sample stands for; None (= uniform) if t is unusable."""
    if t is None or t.size != v.size or t.size < 2: return None
    d = np.gradient(t)
    ok = np.isfinite(d) & (d > 0)
    if not ok.any(): return None
    return np.where(ok, d, np.median(d[ok]))

def w1_load_case(t_real, v_real, t_pred, v_pred):
    if v_real.size < 4 or v_pred.size < 4: return np.nan     # too short to be a distribution
    return float(wasserstein_distance(v_real, v_pred,
                                      u_weights=_dt_weights(t_real, v_real),
                                      v_weights=_dt_weights(t_pred, v_pred)))

def _finite_curve(g):
    t = g['t_minutes'].to_numpy(float); v = g['value'].to_numpy(float)
    m = np.isfinite(t) & np.isfinite(v)
    return t[m], v[m]

def paired_long(df_curves, real_series, pred_series, method_label):
    # One row per (sensor, case) present in BOTH the real and predicted series,
    # carrying every ACTIVE paired metric as its own column.
    rows = []
    for sen, sg in df_curves.groupby('sensor'):
        real_by = {c: g.sort_values('t_minutes')
                   for c, g in sg[sg['series'] == real_series].groupby('case_id')}
        pred_by = {c: g.sort_values('t_minutes')
                   for c, g in sg[sg['series'] == pred_series].groupby('case_id')}
        for cid, rg in real_by.items():
            pg = pred_by.get(cid)
            if pg is None or pg.empty: continue
            t_r, v_r = _finite_curve(rg)
            t_p, v_p = _finite_curve(pg)
            rec = {'sensor': sen, 'case_id': cid, 'method': method_label}
            if INCLUDE_TIME:
                rec['time'] = w1_time_case(t_r, v_r, t_p, v_p)
            if INCLUDE_W1_LOAD:
                rec['w1_load'] = w1_load_case(t_r, v_r, t_p, v_p)
            # keep the row if at least one metric survived its guards
            if any(np.isfinite(rec.get(f, np.nan)) for f in PAIRED_FEATURES):
                rows.append(rec)
    return rows

In [12]:
# ── Resolve process-type -> concrete simulation mode per process ─────────────
pe = pd.read_parquet(RUN / 'process_eval_results.parquet')
# NOTE: no display-split prefix is needed here — every column read from `pe` in
# this cell drives a SELECTION, and selections use _sel_prefix (TRAIN).
def parse_mode(m):
    r = str(m)[len('petri_net_'):] if str(m).startswith('petri_net_') else str(m)
    if r.endswith('_ml_plus_global'): return r[:-len('_ml_plus_global')], 'ml_global'
    if r.endswith('_ml_plus_per_act'): return r[:-len('_ml_plus_per_act')], 'ml_local'
    return r, 'baseline'
pe['model'], pe['time_pred'] = zip(*pe['mode'].map(parse_mode))
_sel_prefix = SELECTION_SPLIT.lower() + '_'          # selection on TRAIN
_sel_cols = [_sel_prefix + b for b in SELECTION_METRIC_BASES
             if (_sel_prefix + b) in pe.columns]
assert _sel_cols, f'no {_sel_prefix}* selection columns found'
# 'combined' is the pipeline's own VERDICT row (mode 'petri_net_combined'), not a
# miner — it carries the winning miner's conformance scores, so leaving it in the
# candidate pool makes it tie with the miner it was selected from and win the
# argmax on name order. mode_for() then points at petri_net_combined/, which the
# simulation never writes, and every Combined-best row is silently dropped.
# Same exclusion as in 03_results_process.ipynb, and for the same reason.
_cands = sorted(set(pe['model'].unique()) - {'alpha', 'budget', 'combined'})
assert _cands, 'no miner candidates left after excluding alpha/budget/combined'
# Prefer the verdict the pipeline recorded during TRAINING; fall back to
# re-deriving it for runs made before petri_net_combined was written.
_pipeline_choice = {}
if 'selected_mining_algorithm' in pe.columns:
    _cmb = pe[(pe['model'] == 'combined') & pe['selected_mining_algorithm'].notna()]
    _pipeline_choice = (_cmb.groupby('process')['selected_mining_algorithm']
                            .first().to_dict())
best_miner = {}
for proc, g in pe.groupby('process'):
    if proc in _pipeline_choice:
        best_miner[proc] = _pipeline_choice[proc]
        continue
    gc = g[g['model'].isin(_cands)]
    _sc = gc.groupby('model')[_sel_cols].mean().mean(axis=1) if not gc.empty else None
    best_miner[proc] = (None if _sc is None else
                        (_sc.idxmax() if SELECTION_HIGHER_IS_BETTER else _sc.idxmin()))
print(f'Combined-best selected by '
      f'{"pipeline, on TRAIN" if _pipeline_choice else f"notebook, on {SELECTION_SPLIT.upper()}"} '
      f'| miner candidates: {_cands}')
for _p, _m in best_miner.items():
    print(f'  {_p}: {_m}')
_TIME_SUFFIX = {'baseline':'', 'ml_global':'_ml_plus_global', 'ml_local':'_ml_plus_per_act'}
# DURATION_MODE='best_by_mae' picks the time-prediction variant per process by
# DURATION_SELECT_METRIC (activity-duration MAE). That is a SELECTION, so it
# reads the TRAIN column -- the tables below report the chosen variant on test,
# and picking it on test would be selecting on the evaluation split (same rule
# as SELECTION_SPLIT above). MAE rather than WAPE: the comparison is always
# within one process, where the scale is constant, so the normalisation WAPE
# adds buys nothing and MAE stays in interpretable minutes.
_dur_sel_col = _sel_prefix + DURATION_SELECT_METRIC
assert _dur_sel_col in pe.columns, f'{_dur_sel_col} missing'
def model_for(proc, ptype):
    return {'Alpha':'alpha','Budget':'budget'}.get(ptype) or best_miner.get(proc)
def mode_for(proc, ptype):
    model = model_for(proc, ptype)
    if model is None: return None
    if DURATION_MODE in _TIME_SUFFIX: tp = DURATION_MODE
    else:
        sub = pe[(pe['process']==proc)&(pe['model']==model)]
        tp = (sub.loc[sub[_dur_sel_col].idxmin(), 'time_pred']
              if not sub.empty and sub[_dur_sel_col].notna().any() else 'baseline')
    return f'petri_net_{model}{_TIME_SUFFIX[tp]}'

Combined-best selected by pipeline, on TRAIN | miner candidates: ['heuristic']
  process_1: heuristic
  process_2: heuristic
  process_3: heuristic
  process_4_1: heuristic
  process_4_2: heuristic
  process_5: heuristic


In [13]:
# ── Build unified feature table (real + every method) ────────────────────────
active_ptypes = [t for t in ['Alpha','Combined-best','Budget'] if PROCESS_TYPES.get(t)]
active_extra  = [s for s,lbl in SCHEDULE_SERIES.items() if EXTRA_METHODS.get(lbl)]
METHOD_ORDER  = (['Baseline'] if SHOW_BASELINE else []) + active_ptypes \
                + [SCHEDULE_SERIES[s] for s in active_extra]

all_rows, time_rows = [], []
for proc in sorted(pe['process'].unique()):
    for ptype in active_ptypes:
        mode = mode_for(proc, ptype)
        fp = RUN/'complete_curve_eval_results'/proc/(mode or '')/f'predicted_curves{_curve_suffix}.parquet'
        # Warn instead of skipping quietly: a mode that resolves to a directory the
        # simulation never wrote drops the method from every table, and an empty
        # row is indistinguishable from a method that was simply switched off.
        if not mode or not fp.exists():
            print(f'  ⚠️ {ptype} curves missing for {proc} ({mode}) — row will be empty')
            continue
        d = pd.read_parquet(fp)
        recs = features_long(d[d['series']=='predicted'], ptype)
        for r in recs: r['process']=proc
        all_rows += recs
        # Paired metrics: compare each simulated case against its own real counterpart
        if INCLUDE_PAIRED:
            for r in paired_long(d, 'real', 'predicted', ptype): r['process']=proc; time_rows.append(r)
    # Baseline: same simulation mode as BASELINE_PROCESS_TYPE, naive per-sensor
    # median curve. Reuses the mode's own 'real' series for the paired metrics.
    if SHOW_BASELINE:
        bmode = mode_for(proc, BASELINE_PROCESS_TYPE)
        bfp = RUN/'complete_curve_eval_results'/proc/(bmode or '')/f'predicted_curves{_baseline_suffix}.parquet'
        if bmode and bfp.exists():
            db = pd.read_parquet(bfp)
            recs = features_long(db[db['series']=='predicted'], 'Baseline')
            for r in recs: r['process']=proc
            all_rows += recs
            if INCLUDE_PAIRED:
                for r in paired_long(db, 'real', 'predicted', 'Baseline'): r['process']=proc; time_rows.append(r)
        else:
            print(f'  ⚠️ Baseline curves missing for {proc} ({bmode}) — row will be empty')

    sfp = RUN/'schedule_profile_eval_results'/proc/'predicted_curves.parquet'
    if not sfp.exists(): continue
    sdf = pd.read_parquet(sfp)
    for r in (rr for rr in features_long(sdf[sdf['series']=='real'], 'real')): r['process']=proc; all_rows.append(r)
    for s in active_extra:
        recs = features_long(sdf[sdf['series']==s], SCHEDULE_SERIES[s])
        for r in recs: r['process']=proc
        all_rows += recs
        if INCLUDE_PAIRED:
            for r in paired_long(sdf, 'real', s, SCHEDULE_SERIES[s]): r['process']=proc; time_rows.append(r)
feat = pd.DataFrame(all_rows)
tim  = pd.DataFrame(time_rows)

# restrict to sensors common to real + every method within each process
keep, keep_t = [], []
for proc, g in feat.groupby('process'):
    sets = [set(g[g.series==m]['sensor'].unique()) for m in (['real']+METHOD_ORDER) if m in set(g.series)]
    common = set.intersection(*sets) if sets else set()
    keep.append(g[g['sensor'].isin(common)])
    if not tim.empty:
        gt = tim[tim['process']==proc]
        keep_t.append(gt[gt['sensor'].isin(common)])
feat = pd.concat(keep, ignore_index=True)
tim  = pd.concat(keep_t, ignore_index=True) if keep_t else tim
print('feat rows:', len(feat), '| paired (W1) case rows:', len(tim), '| methods:', METHOD_ORDER)
# A method that reached zero rows will never appear in any table below.
_missing = [m for m in METHOD_ORDER if m not in set(feat['series'])]
if _missing:
    print(f'⚠️ NO DATA for: {_missing} — these methods are absent from every table/plot')

  ⚠️ Baseline curves missing for process_1 (petri_net_budget_ml_plus_per_act) — row will be empty
  ⚠️ Baseline curves missing for process_2 (petri_net_budget_ml_plus_per_act) — row will be empty
  ⚠️ Baseline curves missing for process_3 (petri_net_budget_ml_plus_per_act) — row will be empty
  ⚠️ Baseline curves missing for process_4_1 (petri_net_budget_ml_plus_per_act) — row will be empty
  ⚠️ Baseline curves missing for process_4_2 (petri_net_budget_ml_plus_per_act) — row will be empty
  ⚠️ Baseline curves missing for process_5 (petri_net_budget_ml_plus_global) — row will be empty
feat rows: 16345 | paired (W1) case rows: 0 | methods: ['Baseline', 'Alpha', 'Combined-best', 'Budget', 'Schedule-direct', 'Profile-generator']
⚠️ NO DATA for: ['Baseline'] — these methods are absent from every table/plot


In [14]:
# ── One row per (process, case, sensor, method, feature) — the raw grid ──────
# AGGREGATION UNIT = (process, case, sensor). Previously the unit was the
# (process, sensor) cell: within a cell the feature's distribution over cases
# for `real` was compared to the one for the method via an *unpaired*
# Wasserstein distance, and the cases were consumed there. Two consequences
# made that a poor choice:
#   * a method could match the marginal distribution while being wrong on every
#     individual case (the distributions overlap, the pairs don't), and
#   * the median was over ~71 cells, weighted by SENSOR COUNT, so process_5
#     (27 sensors, 16 cases) carried 38% of every number while process_1
#     (90 cases) carried 1.4%.
# Now each case is compared to its OWN real counterpart and kept as its own
# row, so every (process, case, sensor) is one unit of the median — and the
# per-unit values survive, which is what a bootstrap CI needs.
#
# Value features: paired relative error |f(pred) - f(real)| / mean|f(real)|,
# normalised per sensor by the same scale as before so magnitudes stay
# comparable across sensors and the degenerate-scale guard still applies.
# 'time' is already a paired per-case quantity and enters unchanged.
records = []
for (proc, sen), g in feat.groupby(['process', 'sensor']):
    real_g = g[g.series == 'real'].drop_duplicates('case_id').set_index('case_id')
    for feature in VALUE_FEATURES:
        r = real_g[feature].dropna()
        if len(r) < 2:
            continue
        scale = np.nanmean(np.abs(r.to_numpy())) + 1e-9
        # Degenerate-scale guard: for zero-inflated energy sensors the per-case
        # 'median' (and occasionally others) is ~0 for every real case, so
        # mean|real| collapses and the relative error explodes. Skip those cells
        # rather than emit meaningless ~1e9 values.
        if scale < 1e-6:
            continue
        for m in METHOD_ORDER:
            q = g[g.series == m].drop_duplicates('case_id').set_index('case_id')[feature].dropna()
            shared = r.index.intersection(q.index)
            if len(shared) == 0:
                continue
            err = (q.loc[shared] - r.loc[shared]).abs() / scale
            for cid, v in err.items():
                records.append({'process': proc, 'sensor': sen, 'case_id': cid,
                                'method': m, 'feature': feature, 'rel_err': float(v)})
units = pd.DataFrame(records)

# Paired metrics: already one value per (process, sensor, case, method).
#   'time'    is dimensionless by construction (time / that case's own real duration).
#   'w1_load' comes out in the sensor's own units (kW), so it is divided by the same
#             kind of scale the value features use -- the sensor's typical real load,
#             mean over real cases of |mean(v)| -- which puts it on the identical
#             'fraction of the typical real value' footing and keeps the Overall
#             average across columns meaningful. Same degenerate-scale guard: sensors
#             whose real load averages ~0 are dropped instead of exploding the ratio.
if INCLUDE_PAIRED and not tim.empty:
    t_units = (tim.melt(id_vars=['process', 'sensor', 'case_id', 'method'],
                        value_vars=[f for f in PAIRED_FEATURES if f in tim.columns],
                        var_name='feature', value_name='rel_err')
                  .dropna(subset=['rel_err']))
    if INCLUDE_W1_LOAD:
        load_scale = (feat[feat.series == 'real']
                          .groupby(['process', 'sensor'])['mean']
                          .apply(lambda s: float(np.nanmean(np.abs(s.to_numpy())))))
        _idx = pd.MultiIndex.from_arrays([t_units['process'], t_units['sensor']])
        scale = pd.Series(load_scale.reindex(_idx).to_numpy(), index=t_units.index)
        _is_load = t_units['feature'] == 'w1_load'
        t_units.loc[_is_load, 'rel_err'] = t_units.loc[_is_load, 'rel_err'] / scale[_is_load]
        t_units = t_units[~(_is_load & ~(scale > 1e-6))]          # NaN scale drops too
    units = pd.concat([units, t_units[['process', 'sensor', 'case_id',
                                       'method', 'feature', 'rel_err']]],
                      ignore_index=True)

print(f'units: {len(units):,} rows | '
      f'{units[["process","case_id","sensor"]].drop_duplicates().shape[0]:,} '
      f'distinct (process, case, sensor)')
print(units.groupby('method').size().reindex(METHOD_ORDER).to_string())


def scorecard(df_units):
    t = (df_units.groupby(['method', 'feature'])['rel_err'].median().unstack('feature')
             .reindex(index=METHOD_ORDER, columns=FEATURES))
    t.columns = [FEAT_LABEL[c] for c in t.columns]
    if SHOW_OVERALL:
        t['Overall'] = t.mean(axis=1)      # simple average across the metric columns
        if SORT_BY_OVERALL:
            t = t.sort_values('Overall')   # best first
    t.index.name = 'Method'
    return t


def style_score(tbl):
    return (tbl.style.format('{:.3f}', na_rep='—')
              .highlight_min(axis=0, props='font-weight:700;background-color:#d6ecff;')
              .set_caption('Median per-case error to real over (process, case, sensor) '
                           '— lower = closer to real'))

units: 67,960 rows | 2,753 distinct (process, case, sensor)
method
Baseline                 NaN
Alpha                13105.0
Combined-best        13565.0
Budget               13765.0
Schedule-direct      13760.0
Profile-generator    13765.0


### What each metric means

Every cell of the grid is one **(process, sensor, method)** pair; the tables show the
**median** of those cells.

**Every column is paired, per case.** For each case a single scalar is computed
from its curve, for both the real and the predicted version. The number is
`|f(pred) - f(real)| / mean|f(real)|` — the error on that case, expressed as a fraction
of the typical real value for that sensor, so it is comparable across sensors with
wildly different units. Sensors where `mean|f(real)|` collapses to ~0 (zero-inflated)
are dropped rather than reported as huge nonsense.

| Column | Per-curve scalar | Reads as |
|---|---|---|
| `Total` | `sum(v)` | total energy drawn over the case — gets the **overall consumption** right |
| `Peak` | `max(v)` | highest instantaneous load — matters for **peak demand / connection sizing** |
| `Mean` | `mean(v)` | average load level over the case |
| `Median` | `median(v)` | *typical* load level, robust to spikes; for on/off sensors this is essentially the **idle / base level** |
| `Std` | `std(v)` | how much the load swings within a case — **amplitude of the dynamics** |
| `AC1` | `corr(v_t, v_{t-1})` | lag-1 autocorrelation: **persistence/smoothness**. ~1 = smooth ramps, ~0 = noisy sample-to-sample. Low error here = the method reproduces how *gradually* load changes |
| `Roughness` | `mean|Δv|` | mean absolute step between consecutive samples, i.e. `mean(|v[i+1] - v[i]|)` — **jaggedness / switching intensity**, in kW per sample. It is the average size of a minute-to-minute change: a smooth ramp barely moves per minute (low), a load that slams between idle and full every few minutes moves a lot (high). It says *how fast the load moves*, where `Std` says *how far it swings* — a slow sine and a fast square wave can share a `Std` but never a `Roughness`. So a method that hits the right amplitude by sprinkling high-frequency noise is caught only here |
| `Zero-frac` | share of samples within 2 % of the curve's own range above its minimum | **duty cycle**: fraction of the case spent at the floor/idle level vs. actually running |

`AC1` and `Roughness` are the two *shape* metrics (are the dynamics realistic?);
`Total`/`Peak`/`Mean`/`Median`/`Std` are *level* metrics (is the magnitude
realistic?); `Zero-frac` is a *duty-cycle* metric. Lower is better
everywhere.

## 0 · Evaluation counts per method

How many evaluations each row of the scorecards below is a median over. The
scorecards only compare like with like if every method is scored on the **same**
(process, case, sensor) units; any shortfall is flagged.

In [15]:
# ── Evaluations behind every number below ────────────────────────────────────
# One unit of every median in the scorecards = one (process, case, sensor) row
# of `units`, per feature. Methods are only comparable if they are scored on the
# same units, so the population is counted here, before any scorecard: the
# per-feature columns are the values that actually enter each median (a sensor
# dropped by the degenerate-scale guard, or a case a method never simulated,
# shows up as a shortfall there).
_UKEY = ['process', 'case_id', 'sensor']

# Equal counts are necessary but not sufficient: two methods can hold the same
# number of units without holding the same ones, so the units every method has
# are intersected explicitly and reported as their own column.
_sets   = {m: set(map(tuple, g[_UKEY].drop_duplicates().to_numpy()))
           for m, g in units.groupby('method')}
_common = set.intersection(*_sets.values()) if _sets else set()
_extra  = {m: len(s - _common) for m, s in _sets.items()}

_g = units.groupby('method')
eval_counts = pd.DataFrame({
    'rows':      _g.size(),
    'units':     _g[_UKEY].apply(lambda d: len(d.drop_duplicates())),
    'processes': _g['process'].nunique(),
    'sensors':   _g.apply(lambda d: len(d[['process', 'sensor']].drop_duplicates())),
    'cases':     _g.apply(lambda d: len(d[['process', 'case_id']].drop_duplicates())),
})
_per_feat = units.pivot_table(index='method', columns='feature', values='rel_err',
                              aggfunc='count', fill_value=0)
_per_feat = _per_feat[[f for f in FEATURES if f in _per_feat.columns]]
_per_feat.columns = [FEAT_LABEL.get(c, c) for c in _per_feat.columns]
eval_counts['shared units']   = pd.Series({m: len(s & _common) for m, s in _sets.items()})
eval_counts['outside shared'] = pd.Series(_extra)   # units not every method has
eval_counts = (eval_counts.join(_per_feat).reindex(METHOD_ORDER)
                          .fillna(0).astype(int))
eval_counts.index.name = 'Method'

display(Markdown('### Evaluation counts per method — the population of every median below'))
display(eval_counts)

_DIAG   = ['shared units', 'outside shared']      # diagnostics, not populations
_uneven = [c for c in eval_counts.columns
           if c not in _DIAG and eval_counts[c].nunique() > 1]

if _uneven or any(_extra.values()):
    print('⚠️ methods are NOT scored on the same population — the scorecards below '
          'are not like-for-like:')
    for c in _uneven:
        hi = eval_counts[c].max()
        print(f'   {c}: max {hi}, others -> ' +
              ', '.join(f'{m}={v}' for m, v in eval_counts[c].items() if v != hi))
    print(f'   (process, case, sensor) units shared by every method: {len(_common)}')
    for m, n in _extra.items():
        if n:
            print(f'   {m}: {n} units outside that shared set — they enter this '
                  f'method\'s median but not every other one\'s')
else:
    print(f'✅ like-for-like: every method is scored on the same {len(_common)} '
          f'(process, case, sensor) units')

### Evaluation counts per method — the population of every median below

,rows,units,processes,sensors,cases,shared units,outside shared,Sum,Max,Mean,Std,Roughness
Method,,,,,,,,,,,,
Baseline,0,0,0,0,0,0,0,0,0,0,0,0
Alpha,13105,2621,6,79,224,2580,41,2621,2621,2621,2621,2621
Combined-best,13565,2713,6,79,230,2580,133,2713,2713,2713,2713,2713
Budget,13765,2753,6,79,231,2580,173,2753,2753,2753,2753,2753
Schedule-direct,13760,2752,6,79,231,2580,172,2752,2752,2752,2752,2752
Profile-generator,13765,2753,6,79,231,2580,173,2753,2753,2753,2753,2753


⚠️ methods are NOT scored on the same population — the scorecards below are not like-for-like:
   rows: max 13765, others -> Baseline=0, Alpha=13105, Combined-best=13565, Schedule-direct=13760
   units: max 2753, others -> Baseline=0, Alpha=2621, Combined-best=2713, Schedule-direct=2752
   processes: max 6, others -> Baseline=0
   sensors: max 79, others -> Baseline=0
   cases: max 231, others -> Baseline=0, Alpha=224, Combined-best=230
   Sum: max 2753, others -> Baseline=0, Alpha=2621, Combined-best=2713, Schedule-direct=2752
   Max: max 2753, others -> Baseline=0, Alpha=2621, Combined-best=2713, Schedule-direct=2752
   Mean: max 2753, others -> Baseline=0, Alpha=2621, Combined-best=2713, Schedule-direct=2752
   Std: max 2753, others -> Baseline=0, Alpha=2621, Combined-best=2713, Schedule-direct=2752
   Roughness: max 2753, others -> Baseline=0, Alpha=2621, Combined-best=2713, Schedule-direct=2752
   (process, case, sensor) units shared by every method: 2580
   Alpha: 41 units outsid

## 1 · Aggregated scorecard (all processes)

In [16]:
score_all = scorecard(units)
display(style_score(score_all))

,Sum,Max,Mean,Std,Roughness,Overall
Method,,,,,,
Budget,0.215,0.078,0.072,0.415,0.604,0.277
Combined-best,0.380,0.079,0.073,0.418,0.598,0.310
Alpha,0.448,0.094,0.088,0.491,0.647,0.354
Schedule-direct,0.363,0.098,0.095,0.545,0.766,0.373
Profile-generator,0.356,0.203,0.100,1.225,8.921,2.161
Baseline,—,—,—,—,—,—


## 2 · Per process (separated)

In [17]:
per_process_scores = {}
for proc in sorted(units['process'].unique()):
    s = scorecard(units[units['process']==proc])
    per_process_scores[proc] = s
    display(Markdown(f'### {proc}'))
    display(style_score(s))

### process_1

,Sum,Max,Mean,Std,Roughness,Overall
Method,,,,,,
Combined-best,0.072,0.065,0.110,0.107,0.296,0.130
Budget,0.063,0.064,0.095,0.102,0.360,0.137
Alpha,0.131,0.075,0.201,0.196,0.490,0.218
Schedule-direct,0.297,0.095,0.135,0.210,0.391,0.226
Profile-generator,0.277,0.267,0.134,0.115,2.031,0.565
Baseline,—,—,—,—,—,—


### process_2

,Sum,Max,Mean,Std,Roughness,Overall
Method,,,,,,
Budget,0.347,0.179,0.286,0.224,0.364,0.280
Combined-best,0.330,0.163,0.366,0.228,0.463,0.310
Alpha,0.459,0.210,0.281,0.238,0.394,0.316
Schedule-direct,0.486,0.196,0.769,0.624,0.606,0.536
Profile-generator,0.695,0.236,0.738,0.220,3.601,1.098
Baseline,—,—,—,—,—,—


### process_3

...

,Sum,Max,Mean,Std,Roughness,Overall
Method,,,,,,
Budget,0.221,0.063,0.047,0.548,0.570,0.290
Combined-best,0.421,0.065,0.046,0.564,0.602,0.340
Alpha,0.625,0.067,0.045,0.569,0.627,0.386
Schedule-direct,0.327,0.077,0.066,0.785,0.901,0.431
Profile-generator,0.298,0.202,0.051,1.620,12.140,2.862
Baseline,—,—,—,—,—,—


### process_4_2

,Sum,Max,Mean,Std,Roughness,Overall
Method,,,,,,
Budget,0.373,0.087,0.058,0.394,0.730,0.328
Combined-best,0.512,0.087,0.055,0.394,0.733,0.356
Alpha,0.463,0.119,0.060,0.497,0.742,0.376
Schedule-direct,0.410,0.100,0.087,0.490,0.882,0.394
Profile-generator,0.416,0.116,0.102,1.472,10.397,2.501
Baseline,—,—,—,—,—,—


### process_5

,Sum,Max,Mean,Std,Roughness,Overall
Method,,,,,,
Schedule-direct,0.385,0.099,0.077,0.633,0.519,0.343
Alpha,0.418,0.125,0.096,0.865,1.192,0.539
Combined-best,0.490,0.107,0.086,0.656,1.429,0.553
Budget,0.135,0.107,0.084,0.758,1.968,0.610
Profile-generator,0.390,0.410,0.074,2.598,38.029,8.300
Baseline,—,—,—,—,—,—


## 3 · LaTeX — aggregated + one table per process

In [18]:
# ── LaTeX: table* + minipage, caption on top, note at the bottom ─────────────
# Layout knobs — change these if the table doesn't sit right on the page.
LATEX_MINIPAGE   = '13cm'    # width of the centring minipage
LATEX_NOTE_WIDTH = '13cm'    # width of the footnote parbox below the table
LATEX_TYPE_COL   = '2.0cm'   # 'Type' column width
LATEX_METHOD_COL = '3.4cm'   # 'Method' column width (wraps to 2 lines on its own)
GROUP_BY_TYPE    = True      # group rows under Type; False = flat sort by Overall

# Display names and the Type each method belongs to.
METHOD_TEX = {
    'Baseline':          'Sensor median',
    'Alpha':             'Alpha Petri Net',
    'Combined-best':     'Best Petri Net',
    'Budget':            'Best Petri Net + Budget',
    'Schedule-direct':   'Schedule-direct',
    'Profile-generator': 'Profile-generator',
}
METHOD_TYPE = {
    'Baseline':          'Baseline',
    'Alpha':             'Process model',
    'Combined-best':     'Process model',
    'Budget':            'Process model',
    'Schedule-direct':   'Schedule-based',
    'Profile-generator': 'Schedule-based',
}
TYPE_ORDER = ['Baseline', 'Process model', 'Schedule-based']

METHOD_NOTE = (
    r'\textbf{Sensor median (baseline)}: the median curve of each sensor, reused for every case. '
    r'\textbf{Alpha Petri Net}: net discovered by the alpha miner. '
    r'\textbf{Best Petri Net}: best discovered net per process, selected on the '
    r'training split by the mean of Fitness, Precision, Generalization and Simplicity. '
    r'\textbf{Best Petri Net + Budget}: the same net, with each case generated to match '
    r'its predicted total-duration budget. '
    r'\textbf{Schedule-direct}: curves placed directly on the real schedule. '
    r'\textbf{Profile-generator}: stochastic profile generator. '
    r'The three Petri-net rows use the \textit{Step DTW} curve predictor '
    r'of Evaluation~2.'
)


def _row_order(tbl):
    """Rows grouped by Type (best Overall first inside each group), or flat."""
    if not GROUP_BY_TYPE:
        return list(tbl.index)
    key = 'Overall' if 'Overall' in tbl.columns else tbl.columns[0]
    order = []
    for typ in TYPE_ORDER:
        grp = [m for m in tbl.index if METHOD_TYPE.get(m) == typ]
        order += sorted(grp, key=lambda m: (pd.isna(tbl.loc[m, key]), tbl.loc[m, key]))
    return order + [m for m in tbl.index if m not in order]


def to_latex_score(tbl, caption, label, note_extra=''):
    metric_cols = list(tbl.columns)
    best = {c: tbl[c].dropna().min() for c in metric_cols if tbl[c].notna().any()}

    def cell(m, c):
        v = tbl.loc[m, c]
        if pd.isna(v):
            return '--'
        s = f'{v:.3f}'
        return r'\textbf{' + s + '}' if abs(v - best.get(c, np.inf)) < 1e-9 else s

    order = _row_order(tbl)
    colfmt = (f'p{{{LATEX_TYPE_COL}}}|p{{{LATEX_METHOD_COL}}}|'
              + '|'.join(['c'] * len(metric_cols)))
    head = (r'\textbf{Type} & \textbf{Method} & '
            + ' & '.join(r'\textbf{' + str(c) + '}' for c in metric_cols) + r' \\')

    body = []
    i = 0
    while i < len(order):
        m = order[i]
        typ = METHOD_TYPE.get(m, '')
        span = 1
        if GROUP_BY_TYPE:
            while i + span < len(order) and METHOD_TYPE.get(order[i + span]) == typ:
                span += 1
        for k in range(span):
            mm = order[i + k]
            tcell = (r'\multirow{' + str(span) + r'}{*}{' + typ + '}') if (k == 0 and GROUP_BY_TYPE) \
                    else ('' if GROUP_BY_TYPE else typ)
            body.append(f'{tcell} & {METHOD_TEX.get(mm, mm)} & '
                        + ' & '.join(cell(mm, c) for c in metric_cols) + r' \\')
        if GROUP_BY_TYPE and i + span < len(order):
            body.append(r'\midrule')
        i += span

    return '\n'.join([
        r'\begin{table*}[H]', r'\centering', '',
        f'\\begin{{minipage}}{{{LATEX_MINIPAGE}}}', r'\centering', '',
        r'\captionsetup{', r'    justification=centering,',
        r'    singlelinecheck=false,', r'    format=plain', r'}', '',
        r'\caption{' + caption + '}', r'\label{' + label + '}', '',
        r'\vspace{-0.5em}', '',
        f'\\begin{{tabular}}{{{colfmt}}}', r'\toprule', head, r'\midrule',
        *body, r'\bottomrule', r'\end{tabular}', '',
        r'\vspace{0.5em}', '',
        f'\\parbox{{{LATEX_NOTE_WIDTH}}}{{%', r'\footnotesize',
        METHOD_NOTE + (' ' + note_extra if note_extra else ''),
        '}', '', r'\end{minipage}', '', r'\end{table*}',
    ])


_metric_note = (r'Cells are the median over (process, case, sensor) of the paired per-case '
                r'relative error $|f(\mathrm{pred})-f(\mathrm{real})|/\overline{|f(\mathrm{real})|}$. '
                r'Lower is better; \textbf{bold} = best per column. '
                r'Overall is the average across the metric columns.')

tex_all = to_latex_score(
    score_all,
    caption='Complete energy-profile comparison, all processes.',
    label=f'tab:energy_profile_all_{EXPERIMENT}',
    note_extra=_metric_note)
print('% ===== ALL PROCESSES =====')
print(tex_all)

for proc, s in per_process_scores.items():
    print(f'\n% ===== {proc} =====')
    print(to_latex_score(
        s,
        caption=f'Complete energy-profile comparison for {proc}.'.replace('_', r'\_'),
        label=f'tab:energy_profile_{proc}_{EXPERIMENT}',
        note_extra=_metric_note))

% ===== ALL PROCESSES =====
\begin{table*}[H]
\centering

\begin{minipage}{13cm}
\centering

\captionsetup{
    justification=centering,
    singlelinecheck=false,
    format=plain
}

\caption{Complete energy-profile comparison, all processes.}
\label{tab:energy_profile_all_1000}

\vspace{-0.5em}

\begin{tabular}{p{2.0cm}|p{3.4cm}|c|c|c|c|c|c}
\toprule
\textbf{Type} & \textbf{Method} & \textbf{Sum} & \textbf{Max} & \textbf{Mean} & \textbf{Std} & \textbf{Roughness} & \textbf{Overall} \\
\midrule
\multirow{1}{*}{Baseline} & Baseline & -- & -- & -- & -- & -- & -- \\
\midrule
\multirow{3}{*}{Process model} & Best Petri Net + Budget & \textbf{0.215} & \textbf{0.078} & \textbf{0.072} & \textbf{0.415} & 0.604 & \textbf{0.277} \\
 & Best Petri Net & 0.380 & 0.079 & 0.073 & 0.418 & \textbf{0.598} & 0.310 \\
 & Alpha Petri Net & 0.448 & 0.094 & 0.088 & 0.491 & 0.647 & 0.354 \\
\midrule
\multirow{2}{*}{Schedule-based} & Schedule-direct & 0.363 & 0.098 & 0.095 & 0.545 & 0.766 & 0.373 \\
 & Profile